# 02. 决策树体系

决策树的核心思想是：不断提问“某个特征是否满足某个条件”，把样本一步步切分到更纯净的子区域中。

## 决策树的形式化定义

            决策树是一类递归分割特征空间的非参数模型。它通过一系列条件判断将输入空间划分为若干互不重叠的区域，并在每个区域上输出类别标签或实值预测。

对分类树而言，叶节点输出类别分布；对回归树而言，叶节点输出该区域内目标值的统计汇总，通常是均值。

## 输入、输出与参数化方式

            输入为结构化特征向量，内部参数不是连续权重矩阵，而是一组离散的切分规则：

- 选择哪个特征
- 在哪个阈值处切分
- 左右子树分别如何继续分裂

输出是由输入样本落入的叶节点决定的。

## 结构分解与信息流

            决策树的结构是层次化的条件分裂图。

根节点负责全局第一次分割；中间节点持续在局部区域上做进一步切分；叶节点则对应最终决策区域。每一次切分都会改变样本所在的子空间，因此决策树本质上是一个分段常数模型。

由于每次切分一般沿单一特征轴进行，标准树模型形成的边界往往是轴对齐的分块区域。

## 优化目标与训练机制

            训练过程是离散搜索问题。分类树选择能最大化纯度提升的切分，回归树选择能最大化误差下降的切分。

递归分裂会不断降低训练误差，但也会逐步提高模型方差。因此剪枝、最大深度限制、最小叶节点样本数等机制并不是附属参数，而是决策树泛化控制的核心。

## 计算复杂度、统计性质与工程代价

            决策树训练通常需要在多个特征和阈值组合上搜索最优分裂，代价高于单次线性模型拟合，但推理速度很快，只需沿树路径向下遍历。

从统计性质看，单棵深树偏差低、方差高；浅树偏差高、方差低。这也是后续随机森林和 Boosting 出现的理论背景。

## 与相邻模型的差异

            与线性模型相比，决策树可以天然表达非线性和特征交互，但稳定性更弱。
与神经网络相比，单棵树在表格数据上通常更容易解释，但表达边界不够平滑，且对数据扰动敏感。
决策树最重要的理论地位在于：它既是一个独立模型，也是绝大多数集成树模型的基本构件。

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(10, 6))
ax.axis("off")
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)

nodes = {
    "root": (5, 8.5, "根节点\nx_j <= t ?"),
    "left": (3, 5.8, "左子树\n继续切分"),
    "right": (7, 5.8, "右子树\n继续切分"),
    "ll": (1.8, 2.5, "叶节点\n预测 A"),
    "lr": (4.2, 2.5, "叶节点\n预测 B"),
    "rl": (5.8, 2.5, "叶节点\n预测 C"),
    "rr": (8.2, 2.5, "叶节点\n预测 D"),
}
for key, (x, y, text) in nodes.items():
    w, h = (1.7, 1.0) if "叶节点" in text else (2.0, 1.1)
    rect = plt.Rectangle((x - w/2, y - h/2), w, h, facecolor="#72B7B2", edgecolor="black")
    ax.add_patch(rect)
    ax.text(x, y, text, ha="center", va="center", fontsize=11)

edges = [("root", "left"), ("root", "right"), ("left", "ll"), ("left", "lr"), ("right", "rl"), ("right", "rr")]
for a, b in edges:
    ax.annotate("", xy=nodes[b][:2], xytext=nodes[a][:2], arrowprops=dict(arrowstyle="->", lw=1.6))
ax.set_title("决策树的层次切分结构")
plt.show()

## 先建立直觉

            决策树最像人类做判断的过程。

比如你在判断“要不要买一台电脑”，可能会这样思考：

- 预算高吗？
- 如果预算高，是不是经常打游戏？
- 如果不打游戏，是不是经常跑深度学习？

这其实就是一棵树：每问一个问题，就把样本分到不同分支，直到得到最终结论。

所以决策树的核心不是复杂数学，而是“不断提问，把复杂问题拆成一连串简单判断”。

## 架构是怎么工作的

            从结构上看，决策树由三部分组成：

- 根节点：第一个切分问题
- 中间节点：继续切分
- 叶子节点：最终预测结果

分类树在叶子节点输出类别；回归树在叶子节点输出数值。

决策树的本质是把特征空间切成很多区域。每次切分都在问：

“如果按这个特征、这个阈值切开，左右两边会不会更纯？”

这就是为什么树模型的决策边界常常是块状、阶梯状的。

## 训练时到底发生了什么

            训练决策树并不是“调一个连续公式”，而是在大量候选切分中搜索最优切分。

每一层都会做两件事：

1. 枚举某个特征和阈值
2. 计算切完之后纯度是否提升

分类任务看的是熵或基尼系数；回归任务看的是平方误差能否下降。

树越深，通常训练集拟合越好，但也更容易把噪声当规律学进去，这就是过拟合。
因此剪枝、限制树深、限制叶子样本数，都是在防止树长得过于复杂。

## 什么时候该用它

            决策树适合：

- 表格数据
- 特征之间有明显条件分支关系的问题
- 需要可解释性的问题
- 想快速做出可视化和业务解释的问题

如果你面对的是结构化表格任务，树模型通常比神经网络更值得优先尝试。

## 最常见的误区

            常见误区：

1. **误以为树越深越强**
   深树往往只是训练集更强，不代表测试集更强。

2. **误以为树模型不需要调参**
   `max_depth`、`min_samples_leaf`、`ccp_alpha` 都会显著影响泛化。

3. **误以为决策树一定稳定**
   单棵树对数据扰动其实很敏感，这也是后面集成学习要解决的问题。

## 1. 分类树中的纯度指标

### 熵（Entropy）

$$
H(S) = - \sum_k p_k \log_2 p_k
$$

### 基尼系数（Gini Impurity）

$$
G(S) = 1 - \sum_k p_k^2
$$

In [ ]:
# 兼容当前 Windows 环境：把临时目录固定到用户目录下的 ASCII 路径，
# 避免 scipy / sklearn 在中文工作目录下寻找临时文件时报错。
from pathlib import Path
import os
import warnings

temp_root = Path(os.environ.get("ML_DL_TMP", str(Path.home() / ".ml_dl_notebook_tmp")))
temp_root.mkdir(exist_ok=True)
os.environ["TMP"] = str(temp_root)
os.environ["TEMP"] = str(temp_root)

warnings.filterwarnings("ignore")

import math
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(42)
random.seed(42)

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "Arial Unicode MS", "DejaVu Sans"]



print("临时目录:", temp_root)

In [ ]:
from sklearn.datasets import load_diabetes, load_iris, make_moons
from sklearn.metrics import ConfusionMatrixDisplay, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, plot_tree

X_moon, y_moon = make_moons(n_samples=400, noise=0.25, random_state=42)

moon_models = {
    "max_depth=1": DecisionTreeClassifier(max_depth=1, random_state=42),
    "max_depth=3": DecisionTreeClassifier(max_depth=3, random_state=42),
    "max_depth=6": DecisionTreeClassifier(max_depth=6, random_state=42),
}

for model in moon_models.values():
    model.fit(X_moon, y_moon)

In [ ]:
def plot_boundary(ax, model, X, y, title):
    xx, yy = np.meshgrid(
        np.linspace(X[:, 0].min() - 0.5, X[:, 0].max() + 0.5, 300),
        np.linspace(X[:, 1].min() - 0.5, X[:, 1].max() + 0.5, 300),
    )
    grid = np.c_[xx.ravel(), yy.ravel()]
    prob = model.predict(grid).reshape(xx.shape)
    ax.contourf(xx, yy, prob, alpha=0.3, cmap="coolwarm")
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap="coolwarm", s=35, edgecolor="k")
    ax.set_title(title)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (name, model) in zip(axes, moon_models.items()):
    plot_boundary(ax, model, X_moon, y_moon, f"决策边界: {name}")
plt.tight_layout()
plt.show()

## 2. Iris 分类树

决策树分类模型会递归切分，直到达到最大深度、节点过小或者纯度足够高。

In [ ]:
iris = load_iris(as_frame=True)
X_iris = iris.data
y_iris = iris.target

X_train_iris, X_test_iris, y_train_iris, y_test_iris = train_test_split(
    X_iris, y_iris, test_size=0.25, random_state=42, stratify=y_iris
)

iris_tree = DecisionTreeClassifier(max_depth=3, random_state=42)
iris_tree.fit(X_train_iris, y_train_iris)

fig, ax = plt.subplots(figsize=(18, 10))
plot_tree(
    iris_tree,
    feature_names=iris.feature_names,
    class_names=iris.target_names,
    filled=True,
    rounded=True,
    fontsize=10,
    ax=ax,
)
ax.set_title("Iris 分类树结构图")
plt.show()

In [ ]:
path = DecisionTreeClassifier(random_state=42).cost_complexity_pruning_path(
    X_train_iris, y_train_iris
)
ccp_alphas = path.ccp_alphas

records = []
for alpha in ccp_alphas[:-1]:
    clf = DecisionTreeClassifier(random_state=42, ccp_alpha=alpha)
    clf.fit(X_train_iris, y_train_iris)
    records.append(
        {
            "ccp_alpha": alpha,
            "depth": clf.get_depth(),
            "leaves": clf.get_n_leaves(),
            "train_score": clf.score(X_train_iris, y_train_iris),
            "test_score": clf.score(X_test_iris, y_test_iris),
        }
    )

prune_df = pd.DataFrame(records)
prune_df.head()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].plot(prune_df["ccp_alpha"], prune_df["depth"], marker="o")
axes[0].set_xscale("log")
axes[0].set_title("剪枝参数与树深度")

axes[1].plot(prune_df["ccp_alpha"], prune_df["train_score"], label="train", marker="o")
axes[1].plot(prune_df["ccp_alpha"], prune_df["test_score"], label="test", marker="o")
axes[1].set_xscale("log")
axes[1].set_title("剪枝参数与准确率")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
y_pred_iris = iris_tree.predict(X_test_iris)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

ConfusionMatrixDisplay.from_predictions(
    y_test_iris, y_pred_iris, display_labels=iris.target_names, cmap="Blues", ax=axes[0]
)
axes[0].set_title("Iris 分类树的混淆矩阵")

importance = pd.Series(iris_tree.feature_importances_, index=iris.feature_names).sort_values(
    ascending=True
)
importance.plot(kind="barh", ax=axes[1], color="teal")
axes[1].set_title("特征重要性")

plt.tight_layout()
plt.show()

## 3. 回归树

对于回归问题，叶子节点输出的是该区域内目标值的平均值。

In [ ]:
diabetes = load_diabetes(as_frame=True)
X_reg = diabetes.data
y_reg = diabetes.target

X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)

reg_depths = [2, 4, 6, None]
reg_records = []
reg_models = {}

for depth in reg_depths:
    model = DecisionTreeRegressor(max_depth=depth, random_state=42)
    model.fit(X_train_reg, y_train_reg)
    preds = model.predict(X_test_reg)
    key = f"max_depth={depth}"
    reg_models[key] = model
    reg_records.append(
        {"模型": key, "MSE": mean_squared_error(y_test_reg, preds), "R^2": r2_score(y_test_reg, preds)}
    )

reg_tree_df = pd.DataFrame(reg_records).sort_values("MSE")
reg_tree_df

In [ ]:
best_reg_key = reg_tree_df.iloc[0]["模型"]
best_reg_model = reg_models[best_reg_key]
best_reg_preds = best_reg_model.predict(X_test_reg)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.barplot(data=reg_tree_df, x="模型", y="MSE", ax=axes[0], palette="Oranges_d")
axes[0].tick_params(axis="x", rotation=15)
axes[0].set_title("不同回归树深度的 MSE")

axes[1].scatter(y_test_reg, best_reg_preds, alpha=0.75, color="darkorange")
min_v = min(y_test_reg.min(), best_reg_preds.min())
max_v = max(y_test_reg.max(), best_reg_preds.max())
axes[1].plot([min_v, max_v], [min_v, max_v], linestyle="--", color="black")
axes[1].set_title(f"最佳回归树 ({best_reg_key}) 的预测效果")
axes[1].set_xlabel("真实值")
axes[1].set_ylabel("预测值")

plt.tight_layout()
plt.show()